# Buổi 10 — Notebook 2: Phân lớp Tuyến tính

**Mục tiêu:** Tự cài đặt hồi quy logistic (linear discriminant function) cho:
- **Phần A:** Phân lớp nhị phân (2 lớp) — tự cài đặt từ đầu
- **Phần B:** Phân lớp nhiều lớp — mở rộng với sklearn

**Dữ liệu:**
- Phần A: `sklearn.datasets.load_breast_cancer` (569 mẫu, 30 đặc trưng, 2 lớp)
- Phần B: `sklearn.datasets.load_wine` (178 mẫu, 13 đặc trưng, 3 lớp)

**Ước tính thời gian:** ~60 phút


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from sklearn.datasets import load_breast_cancer, load_wine
from src import preprocessing, phan_lop


## Phần A — Phân lớp nhị phân (Breast Cancer)

Bài toán: phân loại khối u **lành tính (0)** vs **ác tính (1)**.


In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
TEN_LOP = list(data.target_names)
print(f'Shape: {X.shape}  — Lớp: {TEN_LOP}')
print(f'Phân phối nhãn: {dict(zip(*np.unique(y, return_counts=True)))}')


### A.1 Chuẩn bị dữ liệu

Dùng lại `preprocessing.chia_train_test()` và `preprocessing.chuan_hoa()`.


In [ ]:
X_train, X_test, y_train, y_test = preprocessing.chia_train_test(X, y)
_, X_tr_sc, X_te_sc = preprocessing.chuan_hoa(X_train, X_test)

# Thêm cột bias
X_tr_b = phan_lop.them_bias(X_tr_sc)
X_te_b = phan_lop.them_bias(X_te_sc)
print(f'X_tr_b: {X_tr_b.shape}')


### A.2 Hàm sigmoid

**Hàm cần hoàn thiện:** `phan_lop.sigmoid()`

Kiểm tra: `sigmoid(0)` phải là `0.5`, `sigmoid(100)` phải ≈ `1.0`


In [ ]:
print('sigmoid(0)   =', phan_lop.sigmoid(0))      # 0.5
print('sigmoid(100) =', phan_lop.sigmoid(100))    # ≈ 1.0
print('sigmoid(-10) =', phan_lop.sigmoid(-10))    # ≈ 0.0

# Vẽ đường cong sigmoid
z = np.linspace(-8, 8, 200)
plt.figure(figsize=(6, 3))
plt.plot(z, phan_lop.sigmoid(z), linewidth=2)
plt.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0,   color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.title('Hàm Sigmoid')
plt.tight_layout()
plt.show()


### A.3 Huấn luyện Logistic Regression

**Các hàm cần hoàn thiện theo thứ tự:**
1. `phan_lop.xac_suat_binary()`
2. `phan_lop.log_loss()`
3. `phan_lop.gradient_logistic()`
4. `phan_lop.huan_luyen_logistic()`
5. `phan_lop.du_doan_binary()`


In [ ]:
w, ls_loss = phan_lop.huan_luyen_logistic(
    X_tr_b, y_train, alpha=0.1, n_iter=500
)

plt.figure(figsize=(7, 3))
plt.plot(ls_loss)
plt.xlabel('Vòng lặp')
plt.ylabel('Log loss')
plt.title('Logistic Regression — Đường cong hội tụ')
plt.tight_layout()
plt.show()


### A.4 Đánh giá mô hình

**Hàm cần hoàn thiện:** `phan_lop.danh_gia()`


In [ ]:
y_pred = phan_lop.du_doan_binary(X_te_b, w)
ket_qua = phan_lop.danh_gia(y_test, y_pred, ten_lop=TEN_LOP)


### A.5 So sánh với sklearn


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_sk = LogisticRegression(max_iter=1000, random_state=42)
lr_sk.fit(X_tr_sc, y_train)
y_pred_sk = lr_sk.predict(X_te_sc)

from sklearn.metrics import accuracy_score
print(f'Độ chính xác (cài đặt thủ công): {accuracy_score(y_test, y_pred):.4f}')
print(f'Độ chính xác (sklearn)          : {accuracy_score(y_test, y_pred_sk):.4f}')


> **Câu hỏi A.1:** Thay đổi `nguong` trong `du_doan_binary()` từ 0.5 xuống 0.3.
> Precision và Recall thay đổi thế nào? Khi nào nên dùng ngưỡng thấp hơn?


## Phần B — Phân lớp nhiều lớp (Wine Dataset)

Bài toán: phân loại 3 giống nho theo đặc tính hóa học.


In [ ]:
wine = load_wine()
Xw, yw = wine.data, wine.target
TEN_LOP_WINE = list(wine.target_names)
print(f'Shape: {Xw.shape}  — Lớp: {TEN_LOP_WINE}')
print(f'Phân phối: {dict(zip(*np.unique(yw, return_counts=True)))}')


In [ ]:
Xw_train, Xw_test, yw_train, yw_test = preprocessing.chia_train_test(Xw, yw)
_, Xw_tr_sc, Xw_te_sc = preprocessing.chuan_hoa(Xw_train, Xw_test)
print(f'Train: {Xw_tr_sc.shape[0]}  |  Test: {Xw_te_sc.shape[0]}')


### B.1 Huấn luyện mô hình nhiều lớp

**Hàm cần hoàn thiện:** `phan_lop.huan_luyen_da_lop()`

Gợi ý: Dùng `sklearn.linear_model.LogisticRegression` với `multi_class='multinomial'`.


In [ ]:
model_wine = phan_lop.huan_luyen_da_lop(Xw_tr_sc, yw_train)
print('Mô hình:', model_wine)
print('Số lớp :', model_wine.classes_)


### B.2 Đánh giá


In [ ]:
yw_pred = model_wine.predict(Xw_te_sc)
ket_qua_wine = phan_lop.danh_gia(yw_test, yw_pred, ten_lop=TEN_LOP_WINE)


### B.3 Trực quan hóa — PCA 2D

Chiếu dữ liệu xuống 2 chiều bằng PCA để thấy đường biên phân loại.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
Xw_2d = pca.fit_transform(Xw_te_sc)
yw_2d_pred = model_wine.predict(Xw_te_sc)

colors = ['#e74c3c', '#2ecc71', '#3498db']
markers = ['o', 's', '^']
plt.figure(figsize=(7, 5))
for k in range(3):
    idx = yw_test == k
    plt.scatter(Xw_2d[idx, 0], Xw_2d[idx, 1],
                c=colors[k], marker=markers[k],
                label=TEN_LOP_WINE[k], edgecolors='k', linewidths=0.3, s=50)
    # Đánh dấu mẫu bị dự đoán sai
    wrong = idx & (yw_2d_pred != yw_test)
    plt.scatter(Xw_2d[wrong, 0], Xw_2d[wrong, 1],
                c='black', marker='x', s=80, linewidths=1.5)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Wine — PCA 2D (X = mẫu dự đoán sai)')
plt.legend()
plt.tight_layout()
plt.show()


## Câu hỏi tổng kết

1. Sự khác biệt giữa **Batch GD** ở notebook 1 và **GD cho logistic regression** ở đây là gì?
2. Chiến lược **One-vs-Rest (OvR)** và **Softmax (Multinomial)** khác nhau thế nào?
3. Nếu tập dữ liệu mất cân bằng (imbalanced), metric nào phù hợp hơn: Accuracy hay F1?

---
*Tiếp theo: `03_SVM.ipynb`*
